# ADP Platform Quickstart

This notebook walks through the complete Athena Data Platform (ADP) workflow — from initialization to feature exploration and visualization.

**What you'll learn:**
- How to initialize the platform and run the data pipeline
- Discovering datasets, snapshots, and feature sets
- Loading data with Polars lazy evaluation
- Running SQL queries with DuckDB
- Visualizing price data and indicators

**Prerequisites:** `pip install -e ".[dev]"` from the project root

**Dataset:** BTCUSDT 1-second OHLCV candles (3,600 rows)

## 1. Pipeline Setup

Initialize the platform and run the full pipeline: ingest → snapshot → features.
These commands are idempotent — safe to re-run.

In [ ]:
import subprocess
import os
from pathlib import Path

# Navigate to the project root (parent of notebooks/)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
elif not Path("config/datasets.yaml").exists():
    # Search upward for the project root
    root = Path.cwd()
    while not (root / "config" / "datasets.yaml").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)

# Initialize the platform (creates directories + SQLite DB)
subprocess.run(["adp", "init"], capture_output=True, check=True)
print("Platform initialized.")

In [ ]:
# Step 1: Ingest raw CSV into normalized Parquet format
subprocess.run(["adp", "ingest", "ohlcv_btcusdt", "--force"], capture_output=True, check=True)
# Step 2: Create an immutable, versioned snapshot of the ingested data
subprocess.run(["adp", "snapshot", "create", "ohlcv_btcusdt"], capture_output=True, check=True)
# Step 3: Build analytical features (SMAs, volatility, returns, VWAP) from the snapshot
subprocess.run(["adp", "features", "build", "ohlcv_btcusdt", "candle_factors"], capture_output=True, check=True)
print("Pipeline complete: ohlcv_btcusdt -> candle_factors")

## 2. Discovering Your Data

Use the ADP Python API to explore what datasets, snapshots, and feature sets are available.

In [ ]:
import polars as pl
from adp import (
    list_datasets,
    list_snapshots,
    list_feature_sets,
    load_dataset,
    load_features,
    query_dataset,
    query_features,
)

# Display settings
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(50)

print("=== Registered Datasets ===")
print(list_datasets())

print("\n=== OHLCV Snapshots ===")
print(list_snapshots("ohlcv_btcusdt"))

print("\n=== OHLCV Feature Sets ===")
print(list_feature_sets("ohlcv_btcusdt"))

## 3. Loading Data with Lazy Evaluation

`load_dataset()` returns a Polars **LazyFrame** — no data is loaded into memory until you call `.collect()`. This means you can inspect the schema and build query plans without touching the Parquet files.

In [ ]:
# Load dataset as LazyFrame
lf = load_dataset("ohlcv_btcusdt")

# Inspect schema without loading data
print("Schema (no data loaded yet):")
print(lf.collect_schema())

# Collect just the first 5 rows
print("\nFirst 5 rows:")
print(lf.head(5).collect())

# Descriptive statistics
df = lf.collect()
print(f"\nShape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print("\nDescriptive statistics:")
print(df.describe())

## 4. Loading Features

Features are analytical columns computed from the normalized snapshot. The `candle_factors` feature set adds 7 columns: rolling volatility, two SMAs, EWMA, returns, log returns, and VWAP.

In [ ]:
# Load features as LazyFrame, then collect
feat_df = load_features("ohlcv_btcusdt", "candle_factors").collect()

print(f"Shape: {feat_df.shape}")
print(f"\nColumns: {feat_df.columns}")

# Show features alongside price data
print("\nFeature preview (rows 50-55, after rolling windows fill):")
print(
    feat_df.select(
        ["timestamp", "close", "sma_10", "sma_50", "ewma_20", "rolling_vol_5", "vwap"]
    ).slice(50, 5)
)

# Feature statistics
print("\nFeature statistics (non-null rows):")
print(
    feat_df.select(
        ["rolling_vol_5", "sma_10", "ewma_20", "close_returns", "vwap"]
    ).describe()
)

## 5. SQL Analysis with DuckDB

Use `query_dataset()` and `query_features()` to run SQL directly over Parquet files via DuckDB. The dataset table is named `dataset`, and the features table is named `features`.

In [ ]:
# 5-minute OHLCV summary
result = query_dataset(
    "ohlcv_btcusdt",
    """
    SELECT
        CAST(FLOOR(EXTRACT(MINUTE FROM timestamp) / 5) * 5 AS INTEGER) AS minute_group,
        COUNT(*) AS ticks,
        ROUND(AVG(volume), 4) AS avg_volume,
        ROUND(MAX(high) - MIN(low), 2) AS price_range,
        ROUND(AVG(close), 2) AS avg_close
    FROM dataset
    GROUP BY minute_group
    ORDER BY minute_group
    """,
)
print("=== 5-Minute OHLCV Summary ===")
print(result)

In [ ]:
# Top 10 highest-volatility moments
result = query_features(
    "ohlcv_btcusdt",
    "candle_factors",
    """
    SELECT
        timestamp,
        close,
        rolling_vol_5,
        sma_10,
        close_returns
    FROM features
    WHERE rolling_vol_5 IS NOT NULL
    ORDER BY rolling_vol_5 DESC
    LIMIT 10
    """,
)
print("=== Highest Volatility Moments ===")
print(result)

## 6. Visualizing Price and Indicators

Plot BTCUSDT close price with SMA overlays, rolling volatility, and returns.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend for headless compatibility
import matplotlib.pyplot as plt

# Sort by time for plotting; convert nulls to NaN for consistent matplotlib behavior
plot_df = feat_df.sort("timestamp").fill_null(float("nan"))
ts = plot_df["timestamp"].to_list()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# --- Subplot 1: Close price with moving average overlays ---
axes[0].plot(ts, plot_df["close"].to_list(), label="Close", alpha=0.5, linewidth=0.5)
axes[0].plot(ts, plot_df["sma_10"].to_list(), label="SMA(10)", linewidth=1.5)
axes[0].plot(ts, plot_df["sma_50"].to_list(), label="SMA(50)", linewidth=1.5)
axes[0].plot(ts, plot_df["ewma_20"].to_list(), label="EWMA(20)", linewidth=1.5, linestyle="--")
axes[0].set_ylabel("Price (USD)")
axes[0].legend(loc="upper left")
axes[0].set_title("BTCUSDT Close Price & Moving Averages")
axes[0].grid(True, alpha=0.3)

# --- Subplot 2: Rolling volatility ---
axes[1].plot(ts, plot_df["rolling_vol_5"].to_list(), label="Rolling Vol(5)", color="orange")
axes[1].set_ylabel("Volatility")
axes[1].legend(loc="upper left")
axes[1].set_title("5-Period Rolling Volatility")
axes[1].grid(True, alpha=0.3)

# --- Subplot 3: Close-to-close returns as color-coded bars ---
returns = [0.0 if r != r else r for r in plot_df["close_returns"].to_list()]  # NaN != NaN
colors = ["green" if r >= 0 else "red" for r in returns]
axes[2].bar(ts, returns, color=colors, alpha=0.6)
axes[2].set_ylabel("Return")
axes[2].set_title("Close-to-Close Returns")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Next Steps

Now that you've seen the basics, explore more advanced workflows:

- **02_factor_research_backtest.ipynb** — Build backtest matrices, analyze factor-return relationships, evaluate signal quality
- **03_rfq_trade_analysis.ipynb** — Cross-dataset analysis of RFQ events and trade executions
- **04_funding_rate_risk_monitor.ipynb** — Funding rate risk monitoring and regime detection